This notebook reproduces the main findings of Kendall, Nannicini, and Trebbi's 2015 paper, which looks at how various political campaign messages affect voting. Instead of using Stata, this project uses Python with `pandas` and `statsmodels` to do the same tasks: the data handling, balance checks, and econometric modeling.

First, we bring in all necessary libraries and create a helper function. This lets us import Stata variable labels directly into a pandas DataFrame, making it easier to work with the data.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer

In [ ]:
def get_labels(data):
    stata_reader = pd.read_stata(data, iterator=True)
    labels_dict = stata_reader.variable_labels()
    data_dictionary = pd.DataFrame(columns=["Variable", "Label"], data=list(labels_dict.items()))
    return data_dictionary

We're working with two main datasets_
- **Aggregate Data**, which has precinct-level info on voting results and demographic stats. 
- **Individual Data**, which includes survey responses gathered from a bunch of eligible voters before and after the election.

So, let's go ahead and load both of these datasets along with their variable labels.

In [ ]:
aggregate = pd.read_stata("../Data/Kendall/aggregate.dta") # precint-level
aggregate_labels = get_labels("../Data/Kendall/aggregate.dta")

individual = pd.read_stata("../Data/Kendall/individual.dta") # individual-levels survey
individual_labels = get_labels("../Data/Kendall/individual.dta")




# Data Exploration and Balance Tests

### Aggregate Data Exploration

In [ ]:
# Observations (95)
aggregate.shape

In [ ]:
# Variabel labels
aggregate_labels

In [ ]:
# Data types
aggregate.info()

In [ ]:
# Descriptive Statistis
aggregate.describe()

### Ex-Ante Balancing Tests (Precinct Level) - Replicating Table A1
To check the treatment effects, we first need to make sure the randomization of campaign messages worked. Proper randomization means initial precinct traits are even between treatment and control groups.

We run tests on this by using regressions of those precinct features against the treatment dummy variables. To deal with any spatial linkages, we cluster our standard errors at the polling place, or `building`, level.

In [ ]:
models = []
dep_vars = ["mun11_tenrolled", "giovi", "fiorentina", "saione", "giotto"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm"]
for var in dep_vars:
    reg = smf.ols(formula=f"{var} ~ +{'+'.join(regressors)}", data=aggregate,).fit(cov_type="cluster",
                                                                                  cov_kwds={"groups" : aggregate["building"]})
    models.append(reg)

In [ ]:
table_A1 = Stargazer(models)

labels = ["Valence by Phone", "Valence by mail", "Ideology by phone", "Ideology by mail", "Double by phone", "Double by mail"]
table_A1.rename_covariates(dict(zip(regressors, labels)))
table_A1.custom_columns(["Eligible Voters", "First neighborhood","Second neighborhood", "Third neighborhood", "Fourth neighborhood" ])
table_A1.show_model_numbers(False)
table_A1.title("Ex-Ante Balancing Tests at the Precinct Level")
table_A1.covariate_order(regressors)
table_A1.show_adj_r2 = False
table_A1.show_confidence_intervals(False)
table_A1.show_degrees_of_freedom(False)
table_A1.show_f_statistic = False
table_A1.show_residual_std_err = False
table_A1

The balancing tests confirm that the predetermined variables are well-balanced across the different treatment groups. We can now transition to the individual-level survey data to perform a similar exploration and validation.

### Individual Survey Data Exploration
This dataset contains self-reported characteristics, prior beliefs, and voting declarations from the surveyed voters.

In [ ]:
individual_labels

In [ ]:
individual.shape

In [ ]:
individual.dtypes

In [ ]:
individual.describe()

In [ ]:
individual.vote_f.sum() # 746 people voted

### Ex-Ante Balancing Tests (Individual Level) - Replicating Table A2
Just like with the aggregate data, we must check that individual traits such as gender, age, education, and political leaning are evenly spread in the randomized groups. Standard errors are clustered at the electoral section (or precinct) level here.

In [ ]:
individual["male"] = np.where(individual["s1_gender"] == "female",0,1)
dep_vars = ["male", "over65", "college", "outlf", "white", "other", "centerleft", "house", "press", "tv"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm", "C(s1_date1)"]
models = []

for var in dep_vars:
    reg = smf.ols(formula=f"{var} ~ {'+'.join(regressors)}", data=individual).fit(cov_type="cluster",
                                                                                  cov_kwds={"groups" : individual["section"]})
    models.append(reg)
models

In [ ]:
table_A2 = Stargazer(models)

table_A2.rename_covariates(dict(zip(regressors, labels)))
table_A2.custom_columns(["Male", "Over65", "College Graduate", "Out of labor force", "White collar", "Other occupation", "Center-left", "Home owner", "Read the press", "Watch TV"])
table_A2.show_model_numbers(False)
table_A2.title("Ex-Ante Balancing Tests at the Individual Level")
table_A2.covariate_order(regressors[:6])
table_A2.show_adj_r2 = False
table_A2.show_confidence_intervals(False)
table_A2.show_degrees_of_freedom(False)
table_A2.show_f_statistic = False
table_A2.show_residual_std_err = False
table_A2

## Main Results: Aggregate Treatment Effects (Table 3)

After validating the randomization, we determine the campaign messages' overall causal effects on voting. To do this, we run a regression using precinct-level results like turnout and incumbent shares. We use dummy variables for different treatments, with no message being the control. For accuracy, we cluster standard errors at the polling place level, as explained in the paper's method.

In [42]:
dep_vars = ["mun11_turnout", "mun11_pfanfani_cand", "mun11_pfanfani_parties"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm"]
models = []

for var in dep_vars:
    reg = smf.ols(formula=f"{var} ~ {'+'.join(regressors)}", data=aggregate).fit(cov_type="cluster",
                                                                                 cov_kwds={"groups" : aggregate["building"]})
    models.append(reg)
models


In [43]:
table3 = Stargazer(models)

table3.rename_covariates(dict(zip(regressors, labels)))
table3.custom_columns(["Turnout", "Incumber share", "Incumbent parties"])
table3.show_model_numbers(False)
table3.title("Reduced-Form Individual Estimates, All Groups")
table3.covariate_order(regressors[:6])
table3.show_adj_r2 = False
table3.show_confidence_intervals(False)
table3.show_degrees_of_freedom(False)
table3.show_f_statistic = False
table3.show_residual_std_err = False
table3

The aggregate results indicate that the **valence message delivered via phone** had a significant positive impact on the incumbent's vote share. 

## Main Results: Individual-Level Evidence (Table 4)

We run the same estimates on individual-level survey data to validate our findings. The dependent variables are binary – self-reported turnout and vote choice. So, we use a linear probability model (OLS), clustered by precinct, to stay consistent with our aggregate framework. Plus, we include fixed effects for the survey date.

In [ ]:
dep_vars = ["turnout", "vote_f", "vote_party_f"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm", "C(s1_date1)"]
models = []

for var in dep_vars:
    mod = smf.ols(formula=f"{var} ~ {'+'.join(regressors)}", data=individual)
    reg = mod.fit(cov_type="cluster", cov_kwds={"groups" : individual.loc[mod.data.row_labels, "section"]})
    models.append(reg)
                                                                                
models

In [44]:
table3 = Stargazer(models)

table3.rename_covariates(dict(zip(regressors, labels)))
table3.custom_columns(["Turnout", "Incumber share", "Incumbent parties"])
table3.show_model_numbers(False)
table3.title("Reduced-Form Aggregate Estimates, All Groups")
table3.covariate_order(regressors[:6])
table3.show_adj_r2 = False
table3.show_confidence_intervals(False)
table3.show_degrees_of_freedom(False)
table3.show_f_statistic = False
table3.show_residual_std_err = False
table3

The individual-level estimates are coherent with the aggregate data: **phone calls focusing on the candidate's valence (competence and effort)** proved to be the most effective persuasive campaign tool, significantly increasing the incumbent's vote share. Conversely, ideological messages and messages delivered solely via mail showed negligible or statistically insignificant effects on voting choices.